In [ ]:
import cfg
import matplotlib.pyplot as plt
import numpy as np
from libs.data_request import DataRequest
from libs.node_calibration import NodeCalibrationService

log = cfg.set_logger()

camp_ids = {'FM original': 176}
target_nodes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
campaign_label = 'FM original'
row_plot = 0
x_limits_mhz = (97.0, 99.0)


def print_campaign_parameters(label, campaign_id, params):
    """Print the SDR campaign schedule and configuration in a readable form."""

    print('\n' + '=' * 50)
    print(f'CAMPAIGN: {label.upper()} (ID: {campaign_id})')
    print(f'Name: {params.name}')

    print('\n  [Schedule]')
    for key, value in vars(params.schedule).items():
        print(f'  - {key}: {value}')

    print('\n  [Configuration]')
    for key, value in vars(params.config).items():
        print(f'  - {key}: {value}')
    print('=' * 50)


def print_calibration_status(calibration_result):
    """Summarize which sensors were calibrated and the trust diagnostics emitted by the model."""

    print('\nCALIBRATION STATUS')
    print(f'- model_dir: {calibration_result.model_dir}')
    print(f'- calibrated_sensors: {sorted(calibration_result.calibrated_sensors)}')
    if calibration_result.skipped_sensors:
        print('- skipped_sensors:')
        for sensor_id, reason in sorted(calibration_result.skipped_sensors.items()):
            print(f'    - {sensor_id}: {reason}')

    print('- trust_diagnostics:')
    for sensor_id, sensor_result in sorted(calibration_result.calibrated_sensors.items()):
        trust = sensor_result.trust_summary
        out_of_range = list(trust.out_of_range_feature_names) or ['-']
        print(
            f'    - {sensor_id}: overall_ood={trust.overall_out_of_distribution}, '
            f'frequency_extrapolation={trust.frequency_extrapolation_detected}, '
            f'out_of_range_features={out_of_range}'
        )


def plot_campaign_overlay(ax, nodes_by_name, row_index, spectrum_column, title, y_label):
    """Plot one PSD row per available node on a shared axis."""

    # Each row stores one PSD plus its frequency bounds. Reconstruct the axis
    # locally so the notebook stays independent from plotting side effects.
    for node_name, frame in sorted(nodes_by_name.items()):
        if row_index >= len(frame):
            continue

        row = frame.iloc[row_index]
        power_db = np.asarray(row[spectrum_column], dtype=float)
        start_frequency_mhz = float(row['start_freq_hz']) / 1.0e6
        end_frequency_mhz = float(row['end_freq_hz']) / 1.0e6
        frequencies_mhz = np.linspace(start_frequency_mhz, end_frequency_mhz, len(power_db))
        ax.plot(frequencies_mhz, power_db, label=node_name, linewidth=1.0, alpha=0.8)

    ax.set_title(title)
    ax.set_ylabel(y_label)
    ax.set_xlim(*x_limits_mhz)
    ax.grid(True, alpha=0.42)
    if nodes_by_name:
        ax.legend(loc='best')


def main():
    """Load one campaign through DataRequest, calibrate it, and compare raw versus calibrated PSDs."""

    log.info(f'Starting {cfg.APP_NAME} v{cfg.APP_VERSION} in {cfg.COUNTRY}...')
    dr = DataRequest(log=log, base_url=cfg.API_URL)
    calibration_service = NodeCalibrationService()

    # Keep the existing SDR data-fetch path unchanged: parameters and frames
    # still come from DataRequest and are only adapted after loading.
    params_by_label = {}
    for label, campaign_id in camp_ids.items():
        params = dr.get_campaign_params(campaign_id)
        params_by_label[label] = params
        print_campaign_parameters(label, campaign_id, params)

    df_full = dr.load_campaigns_and_nodes(campaigns=camp_ids, node_ids=target_nodes)
    raw_nodes = df_full[campaign_label]

    calibration_result = calibration_service.calibrate_campaign_frames(
        campaign_label=campaign_label,
        node_frames=raw_nodes,
        campaign_params=params_by_label[campaign_label],
    )
    print_calibration_status(calibration_result)

    calibrated_nodes = {
        sensor_id: sensor_result.calibrated_frame
        for sensor_id, sensor_result in calibration_result.calibrated_sensors.items()
    }

    fig, axes = plt.subplots(2, 1, figsize=(21, 14), sharex=True)
    plot_campaign_overlay(
        axes[0],
        raw_nodes,
        row_plot,
        spectrum_column='pxx',
        title=f'Raw FM Spectrum Comparison (row={row_plot}, campaign={campaign_label})',
        y_label='Power [dBm]',
    )
    plot_campaign_overlay(
        axes[1],
        calibrated_nodes,
        row_plot,
        spectrum_column='calibrated_pxx',
        title=f'Calibrated FM Spectrum Comparison (row={row_plot}, campaign={campaign_label})',
        y_label='Calibrated Power [dBm]',
    )
    axes[1].set_xlabel('Frequency [MHz]')
    plt.tight_layout()
    plt.show()


if __name__ == '__main__':
    main()
